# KIRAVO — Free Kaggle GPU Worker\nEnable T4 x2 + Internet, then run the next cell.\n

In [ ]:
# KIRAVO — Free Kaggle GPU Worker
# Kaggle blocks IPython's os.system() background processes, so this version
# uses Python subprocess + a Flask thread instead of !nohup.

!wget -q https://github.com/Iamkiranofficial/Kiravoo/raw/main/kaggle/kiravo_kaggle_worker.py -O /kaggle/working/kiravo_kaggle_worker.py
!pkill -f cloudflared 2>/dev/null || true
!fuser -k 7860/tcp 2>/dev/null || true
!rm -f /kaggle/working/cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
!chmod +x /kaggle/working/cloudflared

import sys, subprocess, threading, time, re
from pathlib import Path

sys.path.insert(0, "/kaggle/working")
import kiravo_kaggle_worker as worker

# Start the KIRAVO API inside the current notebook session.
server_thread = threading.Thread(
    target=worker.APP.run,
    kwargs={"host": "0.0.0.0", "port": 7860, "threaded": True},
    daemon=True,
)
server_thread.start()

# Wait until Flask has actually bound to port 7860 before opening the tunnel.
import urllib.request
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:7860/health", timeout=1)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("KIRAVO Flask worker did not start on port 7860.")

# Start Cloudflare Quick Tunnel as a real subprocess.
log_path = "/kaggle/working/kiravo-tunnel.log"
log_file = open(log_path, "w")
tunnel = subprocess.Popen(
    ["/kaggle/working/cloudflared", "tunnel", "--url",
     "http://127.0.0.1:7860", "--no-autoupdate"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

# Wait for the public HTTPS URL.
for _ in range(60):
    text = Path(log_path).read_text(errors="ignore") if Path(log_path).exists() else ""
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if match:
        public_url = match.group(0)
        try:
            health = urllib.request.urlopen(public_url + "/health", timeout=10).read().decode()
            print("KIRAVO_WORKER_URL =", public_url)
            print("KIRAVO worker is running:", health)
            break
        except Exception as health_error:
            print("Tunnel exists but worker health check failed:", health_error)
            break
    time.sleep(2)
else:
    print("Tunnel URL not found yet.")
    print(Path(log_path).read_text(errors="ignore")[-4000:])
